# GrayMatter Ablation Study

Train and evaluate all three skip-connection variants of the Hybrid Attention U-Net:

| Variant | `skip_mode` | Description |
|---------|-------------|-------------|
| Plain U-Net | `identity` | Baseline — no skip attention |
| Coordinate Attention | `coord_only` | Triaxial coordinate gating |
| Full CISA | `full` | Coordinate gating + inter-slice convolution |

All variants share the same 4-level 3D U-Net backbone (channels `[32, 64, 128, 256]`), 
preprocessing, training recipe, and data splits.

In [ ]:
import importlib.util, subprocess, sys
for pkg in ["monai", "nibabel", "tqdm"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

## 1. Data Loading

In [ ]:
from pathlib import Path
import json

def _resolve_dataset() -> Path:
    for p in [Path("dataset"), Path("../dataset"), *Path("/kaggle/input").glob("graymatter-dataset*")]:
        if p.is_dir() and (p / "manifests").is_dir():
            return p
    raise FileNotFoundError("Dataset not found. Place dataset/ next to the notebook.")

DATASET_DIR = _resolve_dataset()
print(f"Dataset: {DATASET_DIR.resolve()}")

# Scan folds
manifests_dir = DATASET_DIR / "manifests"
fold_files = sorted(manifests_dir.glob("fold*.json"))
print(f"Folds found: {len(fold_files)}  {[f.stem for f in fold_files]}")

fold_stats = []
for ff in fold_files:
    with open(ff) as f:
        m = json.load(f)
    n_train = len(m.get("training", {}).get("cases", []))
    n_val = len(m.get("validation", {}).get("cases", []))
    fold_stats.append({"fold": ff.stem, "train": n_train, "val": n_val})
    print(f"  {ff.stem}: {n_train} train / {n_val} val")

total_val = sum(s["val"] for s in fold_stats)
print(f"\nTotal unique validation cases across all folds: {total_val} (OOF pool)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

# Show 3 axial slices with labels overlay
mid = img.shape[2] // 2
slices = [mid - 5, mid, mid + 5]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, s in zip(axes, slices):
    ax.imshow(img[0, :, :, s].T, cmap="gray", origin="lower")
    mask = lbl[0, :, :, s].T
    overlay = np.zeros((*mask.shape, 4))
    overlay[mask == 1] = [1, 0, 0, 0.4]   # CA1 = red
    overlay[mask == 2] = [0, 1, 0, 0.4]   # Sub = green
    ax.imshow(overlay, origin="lower")
    ax.set_title(f"Slice {s}")
    ax.axis("off")
fig.suptitle("Sample Case: Axial Slices with Segmentation Overlay")
plt.tight_layout()
plt.show()

## 2. Model Architecture

In [ ]:
from typing import Literal
from pathlib import Path
import json, os, random, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SkipMode = Literal["identity", "coord_only", "full"]
VARIANTS = {"plain_unet": "identity", "coord_attention": "coord_only", "full_cisa": "full"}
FOLDS = [1, 2, 3, 4, 5]

def _gn(ch):
    g = min(8, ch)
    while g > 1 and ch % g != 0: g -= 1
    return nn.GroupNorm(g, ch)

class DoubleConv3D(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv3d(i, o, 3, padding=1, bias=False), _gn(o), nn.ReLU(True),
            nn.Conv3d(o, o, 3, padding=1, bias=False), _gn(o), nn.ReLU(True),
        )
    def forward(self, x): return self.b(x)

class CISA(nn.Module):
    """Coordinate Inter-Slice Attention for skip connections."""
    def __init__(self, ch: int, mode: SkipMode = "coord_only"):
        super().__init__()
        self.mode = mode
        if mode == "identity": return
        mid = max(ch // 8, 8)
        def br():
            return nn.Sequential(nn.Conv3d(ch, mid, 1, bias=False), nn.ReLU(True), nn.Conv3d(mid, ch, 1, bias=False))
        self.bd, self.bh, self.bw = br(), br(), br()
        self.sig = nn.Sigmoid()
        if mode == "full":
            self.inter = nn.Sequential(
                nn.Conv3d(ch, ch, (3,1,1), padding=(1,0,0), groups=ch, bias=False),
                _gn(ch), nn.ReLU(True),
            )
        else:
            self.inter = None

    def forward(self, x):
        if self.mode == "identity": return x
        g = x * self.sig(self.bd(x.mean(2, keepdim=True)))
        g = g * self.sig(self.bh(g.mean(3, keepdim=True)))
        g = g * self.sig(self.bw(g.mean(4, keepdim=True)))
        if self.mode == "coord_only": return g
        return g + self.inter(g)

class HybridAttentionUNet3D(nn.Module):
    def __init__(self, skip_mode: SkipMode = "coord_only"):
        super().__init__()
        c = [32, 64, 128, 256]
        self.e1 = DoubleConv3D(1, c[0])
        self.e2 = DoubleConv3D(c[0], c[1])
        self.e3 = DoubleConv3D(c[1], c[2])
        self.e4 = DoubleConv3D(c[2], c[3])
        self.pool = nn.MaxPool3d(2)
        self.bn = nn.Sequential(DoubleConv3D(c[3], c[3]), nn.Dropout3d(0.1))
        self.s4 = CISA(c[3], mode=skip_mode)
        self.s3 = CISA(c[2], mode=skip_mode)
        self.s2 = CISA(c[1], mode=skip_mode)
        self.s1 = CISA(c[0], mode=skip_mode)
        self.u4 = nn.ConvTranspose3d(c[3], c[3], 2, 2)
        self.u3 = nn.ConvTranspose3d(c[3], c[2], 2, 2)
        self.u2 = nn.ConvTranspose3d(c[2], c[1], 2, 2)
        self.u1 = nn.ConvTranspose3d(c[1], c[0], 2, 2)
        self.d4 = DoubleConv3D(c[3]*2, c[3])
        self.d3 = DoubleConv3D(c[2]*2, c[2])
        self.d2 = DoubleConv3D(c[1]*2, c[1])
        self.d1 = DoubleConv3D(c[0]*2, c[0])
        self.out = nn.Conv3d(c[0], 3, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        b = self.bn(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), self.s4(e4)], 1))
        d3 = self.d3(torch.cat([self.u3(d4), self.s3(e3)], 1))
        d2 = self.d2(torch.cat([self.u2(d3), self.s2(e2)], 1))
        d1 = self.d1(torch.cat([self.u1(d2), self.s1(e1)], 1))
        return self.out(d1)

print("Architecture defined.")

## 3. Data Transforms & Loaders

In [ ]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, ScaleIntensityRanged,
    SpatialPadd, ResizeWithPadOrCropd, EnsureTyped, RandFlipd, AsDiscreted,
)
from monai.data import CacheDataset, DataLoader, list_data_collate

ROI = (48, 64, 48)

def _resolve_dataset() -> Path:
    for p in [Path("dataset"), Path("../dataset"), *Path("/kaggle/input").glob("graymatter-dataset*")]:
        if p.is_dir() and (p / "manifests").is_dir():
            return p
    raise FileNotFoundError("Dataset not found. Place dataset/ next to the notebook.")

DATASET_DIR = _resolve_dataset()

def _find(base, name):
    p = base / name
    if p.exists(): return p
    if name.endswith(".nii.gz"):
        alt = base / name.replace(".nii.gz", ".nii")
        if alt.exists(): return alt
    return p

def _load_cases(manifest, split):
    cases = manifest["training"]["cases"] if split == "train" else manifest["validation"]["cases"]
    out = []
    for c in cases:
        for key in ("image", "label"):
            rel = Path(c[key]).relative_to("dataset")
            c[key] = str(_find(DATASET_DIR / rel.parent, rel.name))
        out.append(c)
    return out

def train_tf():
    return Compose([
        LoadImaged(["image", "label"], image_only=False),
        EnsureChannelFirstd(["image", "label"]),
        Orientationd(["image", "label"], axcodes="RAS"),
        ScaleIntensityRanged("image", a_min=0, a_max=2500, b_min=0, b_max=1, clip=True),
        AsDiscreted("label"),
        SpatialPadd(["image", "label"], spatial_size=ROI),
        ResizeWithPadOrCropd(["image", "label"], spatial_size=ROI),
        EnsureTyped(["image", "label"], track_meta=False),
        RandFlipd(["image", "label"], spatial_axis=i, prob=0.5) for i in range(3)
    ])

def val_tf():
    return Compose([
        LoadImaged(["image", "label"], image_only=False),
        EnsureChannelFirstd(["image", "label"]),
        Orientationd(["image", "label"], axcodes="RAS"),
        ScaleIntensityRanged("image", a_min=0, a_max=2500, b_min=0, b_max=1, clip=True),
        AsDiscreted("label"),
        SpatialPadd(["image", "label"], spatial_size=ROI),
        ResizeWithPadOrCropd(["image", "label"], spatial_size=ROI),
        EnsureTyped(["image", "label"]),
    ])

def load_fold(n):
    with open(DATASET_DIR / "manifests" / f"fold{n}.json") as f:
        m = json.load(f)
    return _load_cases(m, "train"), _load_cases(m, "validation")

print("Transforms ready.")

## 4. Training

In [ ]:
from monai.losses import DiceCELoss
from torch.amp import GradScaler, autocast

MAX_EPOCHS, EARLY_STOP, BATCH_SIZE, LR, SEED = 300, 30, 2, 5e-4, 42
OUTPUT_DIR = Path("ai/results/ablations")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def get_dice(pred, lbl, nc=3):
    dices = []
    for c in range(1, nc):
        p = (pred == c).float().flatten()
        t = (lbl == c).float().flatten()
        dices.append((2 * (p * t).sum() / (p.sum() + t.sum() + 1e-8)).item())
    return np.mean(dices)

def train_fold(variant, skip_mode, fold_num):
    fold_dir = OUTPUT_DIR / variant / f"fold{fold_num}"
    ckpt = fold_dir / "best_model.pth"

    if ckpt.exists():
        with open(fold_dir / "eval_metrics.json") as f:
            prev = json.load(f)
        print(f"  Fold {fold_num} done (DSC: {prev['best_dice']:.4f}) \u2014 skipping")
        return prev

    fold_dir.mkdir(parents=True, exist_ok=True)
    set_seed(SEED + fold_num)

    train_cases, val_cases = load_fold(fold_num)
    train_dl = DataLoader(CacheDataset(train_cases, train_tf(), cache_rate=1.0, num_workers=0),
                          BATCH_SIZE, shuffle=True, collate_fn=list_data_collate)
    val_dl   = DataLoader(CacheDataset(val_cases, val_tf(), cache_rate=1.0, num_workers=0), 1)

    model = HybridAttentionUNet3D(skip_mode=skip_mode).to(device)
    weights = torch.tensor([0.167, 2.83, 3.07], device=device)
    loss_fn = DiceCELoss(include_background=False, to_onehot_y=True, softmax=True,
                         weight=weights, lambda_dice=1.5, lambda_ce=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.SequentialLR(optimizer, [
        torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=10),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS - 10),
    ], [10])
    scaler = GradScaler()

    best_dice, patience, history = 0.0, 0, []

    for epoch in range(MAX_EPOCHS):
        model.train()
        tloss = 0
        for batch in train_dl:
            imgs, lbls = batch["image"].to(device), batch["label"].to(device)
            optimizer.zero_grad()
            with autocast("cuda"):
                loss = loss_fn(model(imgs), lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            tloss += loss.item()
        sched.step()
        tloss /= len(train_dl)

        model.eval()
        vdices = []
        with torch.no_grad():
            for batch in val_dl:
                imgs, lbls = batch["image"].to(device), batch["label"].to(device)
                out = model(imgs).argmax(dim=1, keepdim=True)
                vdices.append(get_dice(out, lbls))
        vdice = np.mean(vdices)
        lr = optimizer.param_groups[0]["lr"]
        history.append({"epoch": epoch+1, "train_loss": tloss, "val_dice": vdice, "lr": lr})

        if vdice > best_dice:
            best_dice, patience = vdice, 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "dice": best_dice}, ckpt)
            if (epoch+1) % 5 == 0 or epoch == 0:
                print(f"    Epoch {epoch+1:3d} | Loss: {tloss:.4f} | DSC: {vdice:.4f} | LR: {lr:.6f} [BEST]")
        else:
            patience += 1
            if (epoch+1) % 25 == 0:
                print(f"    Epoch {epoch+1:3d} | Loss: {tloss:.4f} | DSC: {vdice:.4f} | Pat: {patience}/{EARLY_STOP}")
        if patience >= EARLY_STOP:
            print(f"    Early stopping at epoch {epoch+1}")
            break

    pd.DataFrame(history).to_csv(fold_dir / "training_history.csv", index=False)
    result = {"fold": fold_num, "variant": variant, "skip_mode": skip_mode,
              "best_dice": best_dice, "epochs": len(history)}
    with open(fold_dir / "eval_metrics.json", "w") as f:
        json.dump(result, f, indent=2)
    print(f"    Done: DSC = {best_dice:.4f}")
    return result

print("Training functions ready.")

## 5. Run All Variants

In [ ]:
all_variant_results = {}

for variant, skip_mode in VARIANTS.items():
    print(f"\n{'='*60}")
    print(f"{variant.upper()} (skip_mode={skip_mode})")
    print(f"{'='*60}")

    fold_results = []
    for fold_num in FOLDS:
        print(f"\nFold {fold_num}/{len(FOLDS)}")
        try:
            fold_results.append(train_fold(variant, skip_mode, fold_num))
        except Exception as e:
            print(f"  ERROR: {e}")

    if fold_results:
        dices = [r["best_dice"] for r in fold_results]
        summary = {"variant": variant, "skip_mode": skip_mode,
                   "folds_done": len(fold_results),
                   "mean_dice": float(np.mean(dices)), "std_dice": float(np.std(dices)),
                   "fold_results": fold_results}
        variant_dir = OUTPUT_DIR / variant
        variant_dir.mkdir(parents=True, exist_ok=True)
        with open(variant_dir / "cv_summary.json", "w") as f:
            json.dump(summary, f, indent=2)
        all_variant_results[variant] = summary
        print(f"\n  Mean DSC: {summary['mean_dice']:.4f} \u00b1 {summary['std_dice']:.4f}")

print(f"\nAll variants complete.")

## 6. Out-of-Fold (OOF) Evaluation

In [ ]:
from monai.metrics import DiceMetric, HausdorffDistanceMetric, MeanIoU
from monai.transforms import AsDiscreted
import warnings
warnings.filterwarnings('ignore')

def oof_evaluate_variant(variant, skip_mode):
    all_cases_dsc, all_cases_hd95, all_cases_asd = [], [], []
    all_cases_iou, all_cases_rve = [], []
    per_fold_dsc = {}

    for fold_num in FOLDS:
        fold_dir = OUTPUT_DIR / variant / f"fold{fold_num}"
        ckpt = fold_dir / "best_model.pth"
        if not ckpt.exists():
            print(f"  Fold {fold_num}: no checkpoint, skipping")
            continue

        model = HybridAttentionUNet3D(skip_mode=skip_mode).to(device)
        ckpt_data = torch.load(ckpt, map_location=device, weights_only=True)
        model.load_state_dict(ckpt_data["model"])
        model.eval()

        _, val_cases = load_fold(fold_num)
        val_ds = CacheDataset(val_cases, val_tf(), cache_rate=1.0, num_workers=0)
        val_dl = DataLoader(val_ds, 1, collate_fn=list_data_collate)

        dice_metric = DiceMetric(include_background=False, reduction="none")
        hd_metric = HausdorffDistanceMetric(include_background=False, reduction="none", percentile=95)
        asd_metric = HausdorffDistanceMetric(include_background=False, reduction="none", percentile=None)
        iou_metric = MeanIoU(include_background=False, reduction="none")

        fold_dscs = []
        with torch.no_grad():
            for batch in val_dl:
                imgs = batch["image"].to(device)
                labels = batch["label"].to(device)
                pred = model(imgs).argmax(dim=1, keepdim=True)
                labels_oh = torch.zeros_like(model(imgs))
                for c in range(3):
                    labels_oh[:, c:c+1] = (labels == (c + 1)).float()
                pred_oh = torch.zeros_like(pred.float()).repeat(1, 3, 1, 1, 1)
                for c in range(3):
                    pred_oh[:, c:c+1] = (pred == (c + 1)).float()

                dice_metric(pred_oh, labels_oh)
                hd_metric(pred_oh, labels_oh)
                asd_metric(pred_oh, labels_oh)
                iou_metric(pred_oh, labels_oh)

                case_dice = pred_oh[:, 1:].flatten(2).sum(-1) / (pred_oh[:, 1:].flatten(2).sum(-1) + labels_oh[:, 1:].flatten(2).sum(-1) - (pred_oh[:, 1:] * labels_oh[:, 1:]).flatten(2).sum(-1) + 1e-8)
                fold_dscs.append(case_dice.mean().item())

                pred_np = pred[0, 0].cpu().numpy()
                lbl_np = labels[0, 0].cpu().numpy()
                vol_pred = sum((pred_np == c).sum() for c in [1, 2])
                vol_true = sum((lbl_np == c).sum() for c in [1, 2])
                all_cases_rve.append(abs(vol_pred - vol_true) / (vol_true + 1e-8))

        d = dice_metric.aggregate().cpu().numpy().flatten()
        h = hd_metric.aggregate().cpu().numpy().flatten()
        a = asd_metric.aggregate().cpu().numpy().flatten()
        io = iou_metric.aggregate().cpu().numpy().flatten()

        all_cases_dsc.extend(d.tolist())
        all_cases_hd95.extend(h.tolist())
        all_cases_asd.extend(a.tolist())
        all_cases_iou.extend(io.tolist())
        per_fold_dsc[fold_num] = float(np.mean(fold_dscs))

        dice_metric.reset(); hd_metric.reset(); asd_metric.reset(); iou_metric.reset()
        del model
        torch.cuda.empty_cache()

    return {
        "dsc": float(np.mean(all_cases_dsc)) if all_cases_dsc else None,
        "hd95": float(np.mean(all_cases_hd95)) if all_cases_hd95 else None,
        "asd": float(np.mean(all_cases_asd)) if all_cases_asd else None,
        "iou": float(np.mean(all_cases_iou)) if all_cases_iou else None,
        "rve": float(np.mean(all_cases_rve)) if all_cases_rve else None,
        "per_fold_dsc": per_fold_dsc,
        "n_cases": len(all_cases_dsc),
    }

print("OOF evaluation functions ready.")

In [ ]:
oof_results = {}
for variant, skip_mode in VARIANTS.items():
    print(f"\nOOF evaluation: {variant} (skip_mode={skip_mode})")
    cached = OUTPUT_DIR / variant / "oof_metrics.json"
    if cached.exists():
        with open(cached) as f:
            oof_results[variant] = json.load(f)
        print(f"  Loaded cached OOF ({oof_results[variant]['n_cases']} cases)")
    else:
        res = oof_evaluate_variant(variant, skip_mode)
        oof_results[variant] = res
        with open(cached, "w") as f:
            json.dump(res, f, indent=2)
        print(f"  DSC={res['dsc']:.4f}  HD95={res['hd95']:.2f}  ASD={res['asd']:.2f}  IoU={res['iou']:.3f}  RVE={res['rve']:.3f}  (n={res['n_cases']})")

print("\nAll OOF evaluations complete.")

## 7. Results

In [ ]:
# ── Table 2: Parameter Overhead ─────────────────────────────────────
print("Table 2: Parameter Overhead")
print(f"{'Variant':<28} {'Added params':<16} {'% of 15.0M'}")
print("-" * 56)

backbone = HybridAttentionUNet3D(skip_mode="identity")
base_params = sum(p.numel() for p in backbone.parameters())
del backbone

for variant, skip_mode in VARIANTS.items():
    model = HybridAttentionUNet3D(skip_mode=skip_mode)
    n = sum(p.numel() for p in model.parameters())
    added = n - base_params
    pct = (added / base_params) * 100 if base_params > 0 else 0
    added_str = f"{added:,}" if added > 0 else "0 (reference)"
    label = {
        "plain_unet": "Plain U-Net",
        "coord_attention": "Coord. attention only",
        "full_cisa": "Full CISA",
    }[variant]
    print(f"{label:<28} {added_str:<16} {pct:.2f}%")
    del model

print(f"\nBase backbone: {base_params:,} params (~{base_params/1e6:.1f}M)")
print()

In [ ]:
# ── Table 3: Five-Fold CV DSC ──────────────────────────────────────
print("Table 3: Five-Fold CV DSC")
print(f"{'Variant':<28} {'Mean DSC':<16} {'Std. Dev.'}")
print("-" * 56)

for variant in VARIANTS:
    if variant in all_variant_results:
        s = all_variant_results[variant]
        label = {"plain_unet": "Plain U-Net", "coord_attention": "Coord. attention only", "full_cisa": "Full CISA"}[variant]
        print(f"{label:<28} {s['mean_dice']:<16.4f} {s['std_dice']:.4f}")
print()

In [ ]:
# ── Table 6: Per-Fold OOF DSC ───────────────────────────────────────────────────────────────
print("Table 6: Per-Fold OOF DSC")
VL = {"plain_unet": "Plain U-Net", "coord_attention": "Coord. attn.", "full_cisa": "Full CISA"}
header = f"{'Fold':<10}" + "".join(f"{VL[v]:<18}" for v in VARIANTS)
print(header)
print("-" * (10 + 18 * len(VARIANTS)))

for fold_num in FOLDS:
    row = f"Fold {fold_num:<6}"
    for variant in VARIANTS:
        if variant in oof_results:
            pfd = oof_results[variant].get("per_fold_dsc", {})
            if fold_num in pfd:
                row += f"{pfd[fold_num]:<18.4f}"
            else:
                row += f"{'N/A':<18}"
        else:
            row += f"{'N/A':<18}"
    print(row)

print("-" * (10 + 18 * len(VARIANTS)))
row = f"{'Mean':<10}"
for variant in VARIANTS:
    if variant in oof_results and oof_results[variant]["dsc"] is not None:
        row += f"{oof_results[variant]['dsc']:<18.4f}"
    else:
        row += f"{'N/A':<18}"
print(row)
print()

In [ ]:
# ── Table 4: CV DSC vs OOF Re-Evaluation ───────────────────────
print("Table 4: CV DSC vs OOF Re-Evaluation (n = 260)")
print(f"{'Variant':<28} {'CV DSC':<12} {'OOF DSC':<12} {'Delta':<12} {'%'}")
print("-" * 72)
for variant in VARIANTS:
    cv = all_variant_results.get(variant, {}).get("mean_dice")
    oo = oof_results.get(variant, {}).get("dsc")
    label = {"plain_unet": "Plain 3D U-Net", "coord_attention": "Coord. attention only", "full_cisa": "Full CISA"}[variant]
    if cv is not None and oo is not None:
        delta = oo - cv
        pct = (delta / cv) * 100 if cv != 0 else 0
        print(f"{label:<28} {cv:<12.4f} {oo:<12.4f} {delta:<12.4f} ({pct:+.1f}%)")
    else:
        print(f"{label:<28} {'N/A':<12} {'N/A':<12} {'N/A':<12}")
print()

# ── Table 5: OOF Multi-Metric ──────────────────────────────
print("Table 5: OOF Metrics (n = 260)")
metrics = ["DSC", "HD95 (mm)", "ASD (mm)", "IoU", "RVE"]
keys = ["dsc", "hd95", "asd", "iou", "rve"]
header = f"{'Metric':<14}" + "".join(f"{VL[v]:<18}" for v in VARIANTS)
print(header)
print("-" * (14 + 18 * len(VARIANTS)))
for m, k in zip(metrics, keys):
    row = f"{m:<14}"
    for variant in VARIANTS:
        val = oof_results.get(variant, {}).get(k)
        if val is not None:
            fmt = f"{val:.2f}" if k in ("hd95", "asd") else f"{val:.3f}" if k == "iou" else f"{val:.4f}" if k == "dsc" else f"{val:.3f}"
            row += f"{fmt:<18}"
        else:
            row += f"{'N/A':<18}"
    print(row)
print()

# ── Table 7: Fold-Selection Bias (OOF) ────────────────────
print("Table 7: Fold-Selection Bias: Best-Fold CV DSC vs Pooled OOF DSC")
print(f"{'Variant':<24} {'Best fold':<12} {'Best-fold CV':<16} {'OOF DSC':<12} {'Bias'}")
print("-" * 80)
for variant in VARIANTS:
    cv = all_variant_results.get(variant, {})
    oo = oof_results.get(variant, {})
    if cv and oo and oo.get("dsc") is not None:
        best = max(cv["fold_results"], key=lambda r: r["best_dice"])
        bias = ((best["best_dice"] - oo["dsc"]) / oo["dsc"]) * 100
        label = {"plain_unet": "Plain U-Net", "coord_attention": "Coord. attn. only", "full_cisa": "Full CISA"}[variant]
        print(f"{label:<24} {best['fold']:<12} {best['best_dice']:<16.4f} {oo['dsc']:<12.4f} +{bias:.1f}%")
print()

## 8. Figures

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

VL = {"plain_unet": "Plain U-Net", "coord_attention": "Coord. attn.", "full_cisa": "Full CISA"}
colors = {"plain_unet": "#4C72B0", "coord_attention": "#55A868", "full_cisa": "#C44E52"}

# Figure 2: Fold-Selection Bias
fig, ax = plt.subplots(figsize=(7, 4))
x_pos = np.arange(len(VARIANTS))
cv_best, oof_vals, bias_pcts = [], [], []
for v in VARIANTS:
    cv = all_variant_results.get(v, {})
    oo = oof_results.get(v, {})
    if cv and oo and oo.get("dsc") is not None:
        best = max(cv["fold_results"], key=lambda r: r["best_dice"])
        cv_best.append(best["best_dice"])
        oof_vals.append(oo["dsc"])
        bias_pcts.append(((best["best_dice"] - oo["dsc"]) / oo["dsc"]) * 100)
    else:
        cv_best.append(0); oof_vals.append(0); bias_pcts.append(0)
w = 0.35
ax.bar(x_pos - w/2, cv_best, w, label="Best-Fold CV DSC", color=[colors[v] for v in VARIANTS], alpha=0.85)
ax.bar(x_pos + w/2, oof_vals, w, label="Pooled OOF DSC", color=[colors[v] for v in VARIANTS], alpha=0.45)
for i, b in enumerate(bias_pcts):
    ax.annotate(f"+{b:.1f}%", (x_pos[i], max(cv_best[i], oof_vals[i]) + 0.002), ha="center", fontsize=9)
ax.set_xticks(x_pos)
ax.set_xticklabels([VL[v] for v in VARIANTS])
ax.set_ylabel("DSC")
ax.set_ylim(0.83, 0.89)
ax.set_title("Figure 2: Fold-Selection Bias")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figure2_fold_bias.png", dpi=150)
plt.show()

# Figure 3: Multi-Metric OOF Comparison
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
metric_names = ["DSC", "HD95 (mm)", "ASD (mm)", "IoU"]
metric_keys = ["dsc", "hd95", "asd", "iou"]
for ax, mn, mk in zip(axes, metric_names, metric_keys):
    vals = [oof_results.get(v, {}).get(mk, 0) for v in VARIANTS]
    bars = ax.bar(range(len(VARIANTS)), vals, color=[colors[v] for v in VARIANTS], alpha=0.85)
    ax.set_xticks(range(len(VARIANTS)))
    ax.set_xticklabels([VL[v].split()[0] for v in VARIANTS], fontsize=8)
    ax.set_title(mn)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{val:.3f}", ha="center", va="bottom", fontsize=8)
fig.suptitle("Figure 3: Multi-Metric OOF Comparison")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figure3_oof_metrics.png", dpi=150)
plt.show()

print("Figures saved.")